# Geostore registration

The GFW query endpoint takes a geometry, but territory polygons are too large
to send inline on every request. GFW provides a geostore endpoint: upload a
geometry once, get back an ID, then query by ID from then on.

Two things to establish before writing the pipeline script:

1. whether the endpoint accepts the largest polygons as-is, or whether they
   need simplifying first
2. how long one request takes, to estimate the full run over 3,866 territories

In [1]:
import geopandas as gpd
import os
from dotenv import load_dotenv
import time
import pandas as pd
from pathlib import Path
import requests
import json

load_dotenv("../.env")
API_KEY = os.getenv("GFW_API_KEY")

territories = gpd.read_parquet("../data/processed/territories.parquet")

largest = territories.nlargest(5, "area_ha")
print(largest[["territory_id", "name", "country", "area_ha"]].to_string(index=False))

 territory_id                               name  country      area_ha
         4016                           Yanomami   Brasil 9.537676e+06
         3895                     Vale do Javari   Brasil 8.574051e+06
         4068                     Alto Rio Negro   Brasil 7.980786e+06
        20706 Resguardo Indígena Predio Putumayo Colombia 5.805104e+06
         3952                        Menkragnoti   Brasil 4.949201e+06


Checking the five largest territories first. If the heaviest one is accepted,
everything else will be.

In [2]:
for _, row in largest.iterrows():
    geo = json.dumps(row.geometry.__geo_interface__)
    print(f"{row['name'][:30]:<32} {len(geo)/1024:>8.1f} KB")

Yanomami                           2450.8 KB
Vale do Javari                      759.9 KB
Alto Rio Negro                     1422.9 KB
Resguardo Indígena Predio Putu     1439.4 KB
Menkragnoti                         358.3 KB


Yanomami serialises to 2.4 MB of GeoJSON, which is large for an HTTP request body. Testing it directly is faster than guessing.

In [3]:
row = largest.iloc[0]  # Yanomami
geom = row.geometry.__geo_interface__

resp = requests.post(
    "https://data-api.globalforestwatch.org/geostore",
    headers={"x-api-key": API_KEY, "Content-Type": "application/json"},
    json={"geometry": geom},
    timeout=120,
)

print(resp.status_code)
print(resp.json()["data"]["gfw_geostore_id"])

201
726004de-be98-1460-9deb-057088c2639e


The sample deliberately includes the five largest territories, so the mean is not optimistic. It is also biased for the same reason, which a later cell corrects.

In [4]:
sample = pd.concat([
    largest,
    territories.sample(20, random_state=42)
]).drop_duplicates(subset="territory_id")

results = []
failures = []

for _, row in sample.iterrows():
    t0 = time.time()
    try:
        r = requests.post(
            "https://data-api.globalforestwatch.org/geostore",
            headers={"x-api-key": API_KEY, "Content-Type": "application/json"},
            json={"geometry": row.geometry.__geo_interface__},
            timeout=180,
        )
        elapsed = time.time() - t0
        if r.status_code in (200, 201):
            results.append({
                "territory_id": row["territory_id"],
                "area_ha": row["area_ha"],
                "geostore_id": r.json()["data"]["gfw_geostore_id"],
                "seconds": elapsed,
            })
        else:
            failures.append((row["territory_id"], row["name"], r.status_code, r.text[:150]))
    except Exception as e:
        failures.append((row["territory_id"], row["name"], "EXC", str(e)[:150]))

print(f"succeeded: {len(results)}   failed: {len(failures)}")

succeeded: 25   failed: 0


In [5]:
df = pd.DataFrame(results)
print(df["seconds"].describe().to_string())

print(f"\nBiased estimate for 3,866: {df['seconds'].mean() * 3866 / 60:.0f} min")
print(f"Worst case (max x 3,866):      {df['seconds'].max() * 3866 / 60:.0f} min")

print("\nArea vs time correlation:", df["area_ha"].corr(df["seconds"]).round(3))

for f in failures:
    print(f)

count    25.000000
mean      2.267054
std       2.584976
min       0.628449
25%       0.897216
50%       0.921482
75%       1.453919
max      10.813967

Biased estimate for 3,866: 146 min
Worst case (max x 3,866):      697 min

Area vs time correlation: 0.776


The five largest are 20% of this sample but 0.13% of the population, so the raw mean overstates the total. Removing them gives a usable estimate.

In [6]:
top_ids = set(largest["territory_id"])
typical = df[~df["territory_id"].isin(top_ids)]

print(f"Typical territories: n={len(typical)}, mean={typical['seconds'].mean():.2f}s")
print(f"Unbiased estimate:   {typical['seconds'].mean() * 3866 / 60:.0f} min")

Typical territories: n=20, mean=1.27s
Unbiased estimate:   82 min


In [7]:
out = Path("../data/processed/geostores.csv")
out.parent.mkdir(parents=True, exist_ok=True)

if out.exists():
    print(f"{out} already exists, not overwriting")
else:
    pd.DataFrame(results)[["territory_id", "geostore_id"]].to_csv(out, index=False)
    print(f"saved {len(results)} geostores")

../data/processed/geostores.csv already exists, not overwriting


## Conclusions

- The endpoint accepted a 2.4 MB polygon (HTTP 201). No simplification is needed, so territory boundaries stay exact.
- 25 of 25 requests succeeded, no failures.
- Typical territories take about 1.3 s per request, giving roughly 80 minutes for a sequential run over all 3,866.
- Request time correlates with territory area, so the raw sample mean had to be corrected: the five largest are 20% of this sample but 0.13% of the population.
- The notebook was re-run several times. The correlation ranged from 0.59 to 0.90 and the slowest request from 6.9 s to 10.8 s, but the corrected estimate stayed at 81 to 82 minutes. Network variance affects the tail far more than the median, which is why the corrected figure is the one to trust.

Sequential was chosen over parallel: this is a one-off run, it runs locally, and the code is simpler. The pipeline script writes each row as it goes so an interrupted run can resume.

Implemented in `src/create_geostores.py`.